<a href="https://colab.research.google.com/github/freitasbtt/p4_mackenzie_grupo_11/blob/main/notebooks/etapa_03_implementacao_parcial.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Monitoramento e previsão do emprego formal no Brasil
## Etapa 3 — Implementação parcial — 26/10/2026
**Guilherme da Silva Brevilato · Isabela Kazue Sinoduka · Pedro Henrique Bittencourt de Freitas**  
Universidade Presbiteriana Mackenzie — Projeto Aplicado IV

**Pergunta:** como o saldo de empregos formais evolui ao longo do tempo e em que medida seu histórico permite prever os meses seguintes?

Esta implementação constrói uma série nacional mensal desde janeiro de 2020, explora sua qualidade e comportamento e compara uma referência sazonal com Holt-Winters aditivo. A relação com o ODS 8 é contextual: saldo de vínculos não mede diretamente desemprego, empregabilidade ou qualidade do trabalho.

**Estado do artefato:** código preparado para execução no Colab. As saídas serão geradas com dados reais ao executar as células; nenhum resultado empírico foi preenchido previamente.


## 1. Como executar
1. No Colab, use um ambiente Python e execute as células na ordem.
2. Autorize a montagem do seu Google Drive. Não é necessário projeto de cobrança no BigQuery.
3. Confira os parâmetros. O intervalo padrão é **2020-01 a 2026-08**, fixado para reprodutibilidade; a coleta verifica a existência de cada competência.
4. Execute a coleta. São muitos microdados: ela pode exigir várias sessões, espaço no Drive e tempo de processamento. Os resultados por competência ficam em cache.
5. Execute a exploração e a modelagem, leia as saídas e complete a interpretação final com os resultados observados.
6. Salve o notebook com suas saídas e versione o código no repositório do grupo. Não envie os microdados volumosos ao GitHub.

Se o FTP não estiver acessível, a execução interrompe com erro; ela não substitui dados reais por dados simulados. Os arquivos oficiais `.7z` podem ser colocados manualmente em `data/bronze/AAAAMM/` para reutilização.


In [ ]:
%pip -q install "pandas>=2.2,<3" "numpy>=1.26,<3" "matplotlib>=3.8,<4" "seaborn>=0.13,<0.14" "statsmodels>=0.14,<0.15" "py7zr>=0.22,<2" "pyarrow>=15"


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.3/10.3 MB 67.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 75.5/75.5 kB 6.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 493.4/493.4 kB 29.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 100.8/100.8 kB 7.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.4/52.4 kB 3.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 144.5/144.5 kB 11.7 MB/s eta 0:00:00


In [ ]:
from pathlib import Path
from datetime import datetime, timezone
import ftplib, hashlib, json, re, tempfile, unicodedata, warnings, sys
import importlib.metadata as metadata
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import py7zr
from IPython.display import display, Markdown
from statsmodels.tsa.holtwinters import ExponentialSmoothing
from statsmodels.graphics.tsaplots import plot_acf
from statsmodels.tsa.seasonal import STL
from statsmodels.stats.diagnostic import acorr_ljungbox
from google.colab import drive

drive.mount('/content/drive')
ROOT = Path('/content/drive/MyDrive/Projeto_IV_Mack')
INICIO, FIM = '2020-01', '2026-08'
MESES_TESTE = 12
CHUNKSIZE = 300_000
REPROCESSAR = False  # True recalcula agregados dos arquivos locais; não atualiza o download.
FTP_HOST = 'ftp.mtps.gov.br'
FTP_BASE = '/pdet/microdados/NOVO CAGED'
for pasta in ['data/bronze', 'data/silver', 'data/gold', 'outputs']:
    (ROOT / pasta).mkdir(parents=True, exist_ok=True)
MESES = pd.period_range(INICIO, FIM, freq='M').strftime('%Y%m').tolist()
assert len(MESES) >= MESES_TESTE + 36, 'Use ao menos 36 meses de treino mais o teste.'
sns.set_theme(style='whitegrid')
plt.rcParams.update({'figure.figsize': (12, 4), 'axes.titlesize': 13})
versoes = {p: metadata.version(p) for p in ['pandas','numpy','matplotlib','seaborn','statsmodels','py7zr','pyarrow']}
registro = dict(inicio=INICIO, fim=FIM, teste=MESES_TESTE, recorte='MOV sem FOR/EXC',
                execucao_utc=datetime.now(timezone.utc).isoformat(), python=sys.version, versoes=versoes)
(ROOT/'outputs/ambiente.json').write_text(json.dumps(registro, indent=2), encoding='utf-8')
display(registro)


Mounted at /content/drive


{'inicio': '2020-01',
 'fim': '2026-08',
 'teste': 12,
 'recorte': 'MOV sem FOR/EXC',
 'execucao_utc': '2026-10-18T13:22:28.359138+00:00',
 'python': '3.13.16 (main, Oct  1 2026, 15:40:24) [GCC 13.3.0]',
 'versoes': {'pandas': '2.2.3',
  'numpy': '2.1.3',
  'matplotlib': '3.10.0',
  'seaborn': '0.13.2',
  'statsmodels': '0.14.6',
  'py7zr': '1.1.4',
  'pyarrow': '23.0.1'}}

## 2. Fonte, recorte e limitações
Fonte primária: [MTE — Microdados RAIS e CAGED](https://www.gov.br/trabalho-e-emprego/pt-br/assuntos/estatisticas-trabalho/microdados-rais-e-caged). Os arquivos do Novo CAGED são textos delimitados por ponto e vírgula, disponibilizados em UTF-8 e compactados. A consulta anterior a `microdados_antigos` não é utilizada.

**Simplificação da Etapa 3:** utilizar exclusivamente `CAGEDMOV`. Os arquivos `CAGEDFOR` (fora do prazo) e `CAGEDEXC` (exclusões) não entram nesta implementação. O alvo é, portanto, o **saldo das movimentações do recorte MOV**, e não necessariamente o saldo oficial ajustado mais recente. A incorporação de ajustes exige regras por competência de movimentação e versão da divulgação; não basta concatenar indiscriminadamente os arquivos.

Não há junções ou encodings categóricos nesta versão: a previsão é univariada e nacional. Não serão usados salários ou características pessoais, pois não são necessários ao alvo. A leitura em blocos evita carregar todos os registros em memória. A camada Silver conserva contagens padronizadas por arquivo e mês, em vez de duplicar todos os microdados tratados: esta é uma revisão de armazenamento do pipeline original.

Os arquivos baixados são uma fotografia da fonte na data registrada, não uma reconstrução das versões disponíveis em cada data histórica. Assim, o teste temporal não é uma avaliação rigorosa em tempo real. Não misturamos CAGED antigo e Novo CAGED.


## 3. Extração e rastreabilidade
Cada arquivo original fica preservado na Bronze. O manifesto contém origem, data de obtenção, tamanho e SHA-256. Downloads incompletos usam sufixo temporário. As listagens remotas determinam os nomes dos arquivos; não há tentativa de adivinhar nomes.

Para atualizar uma competência com uma nova versão da fonte, arquive a versão antiga, retire seu manifesto da pasta ativa e use `REPROCESSAR=True`. Uma repetição normal utiliza a fotografia já armazenada.


In [ ]:
def sha256(path):
    h = hashlib.sha256()
    with open(path, 'rb') as f:
        for bloco in iter(lambda: f.read(1024*1024), b''):
            h.update(bloco)
    return h.hexdigest()

def obter_arquivos(mes):
    pasta = ROOT/'data/bronze'/mes
    pasta.mkdir(parents=True, exist_ok=True)
    manifesto = pasta/'manifesto.json'
    if manifesto.exists():
        itens = json.loads(manifesto.read_text(encoding='utf-8'))
        for item in itens:
            p = pasta/item['arquivo']
            if not p.exists() or sha256(p) != item['sha256']:
                raise ValueError(f'Arquivo ausente ou alterado: {p}. Verifique antes de continuar.')
        return itens
    locais = sorted(pasta.glob('CAGEDMOV*.7z'))
    remoto = f'{FTP_BASE}/{mes[:4]}/{mes}'
    origem = 'arquivo local; obtenção anterior não verificada'
    if not locais:
        with ftplib.FTP(FTP_HOST, timeout=180) as ftp:
            ftp.login()
            ftp.cwd(remoto)
            nomes = sorted(n for n in ftp.nlst() if Path(n).name.upper().startswith('CAGEDMOV') and n.lower().endswith('.7z'))
            if not nomes:
                raise FileNotFoundError(f'Nenhum CAGEDMOV encontrado em {remoto}')
            for nome in nomes:
                alvo = pasta/Path(nome).name
                temporario = alvo.with_suffix('.7z.part')
                with open(temporario, 'wb') as f:
                    ftp.retrbinary('RETR '+nome, f.write, blocksize=1024*1024)
                temporario.replace(alvo)
        locais = sorted(pasta.glob('CAGEDMOV*.7z'))
        origem = f'ftp://{FTP_HOST}{remoto}'
    itens = [dict(arquivo=p.name, origem=origem, competencia_pasta=mes,
                  registrado_utc=datetime.now(timezone.utc).isoformat(),
                  bytes=p.stat().st_size, sha256=sha256(p)) for p in locais]
    manifesto.write_text(json.dumps(itens, indent=2), encoding='utf-8')
    return itens


## 4. Validação e preparação
Usamos `competênciamov`/`competenciamov` e `saldomovimentação`/`saldomovimentacao`, normalizando acentos e espaços do cabeçalho. O sinal de movimentação é validado no domínio `+1/-1`: admissão/desligamento. A rotina interrompe se encontrar valores nulos, inválidos ou competências diferentes da pasta, para evitar exclusões silenciosas.

Registros iguais nas colunas selecionadas **não são deduplicados**: diferentes vínculos podem gerar movimentos com os mesmos valores. Não há chave individual suficiente para deduplicação segura. O controle de ingestão por arquivo evita a repetição acidental da mesma entrada.


In [ ]:
def normalizar(texto):
    texto = unicodedata.normalize('NFKD', str(texto).replace('\ufeff',''))
    return re.sub(r'[^a-z0-9]', '', texto.encode('ascii','ignore').decode().lower())

def resumir_txt(path, mes):
    cab = pd.read_csv(path, sep=';', encoding='utf-8-sig', nrows=0)
    mapa = {normalizar(c): c for c in cab.columns}
    necessarias = ['competenciamov', 'saldomovimentacao']
    if not all(c in mapa for c in necessarias):
        raise ValueError(f'Esquema inesperado em {path.name}: {list(cab.columns)}')
    qualidade = dict(arquivo=path.name, mes=mes, registros=0, nulos_competencia=0,
                     nulos_sinal=0, sinais_invalidos=0, competencias_invalidas=0,
                     competencia_divergente=0, admissoes=0, desligamentos=0)
    for bloco in pd.read_csv(path, sep=';', encoding='utf-8-sig', dtype='string',
                             usecols=[mapa[c] for c in necessarias], chunksize=CHUNKSIZE):
        comp = bloco[mapa['competenciamov']].str.strip()
        sinal_original = bloco[mapa['saldomovimentacao']].str.strip()
        sinal = pd.to_numeric(sinal_original, errors='coerce')
        datas = pd.to_datetime(comp, format='%Y%m', errors='coerce')
        qualidade['registros'] += len(bloco)
        qualidade['nulos_competencia'] += int((comp.isna() | comp.eq('')).sum())
        qualidade['nulos_sinal'] += int((sinal_original.isna() | sinal_original.eq('')).sum())
        qualidade['sinais_invalidos'] += int((~sinal.isin([-1,1])).sum())
        qualidade['competencias_invalidas'] += int(datas.isna().sum())
        qualidade['competencia_divergente'] += int((comp.ne(mes).fillna(False)).sum())
        qualidade['admissoes'] += int(sinal.eq(1).sum())
        qualidade['desligamentos'] += int(sinal.eq(-1).sum())
    return qualidade

def processar_mes(mes):
    itens = obter_arquivos(mes)
    assinatura = [i['sha256'] for i in itens]
    cache = ROOT/'data/silver'/f'{mes}.json'
    if cache.exists() and not REPROCESSAR:
        anterior = json.loads(cache.read_text(encoding='utf-8'))
        if anterior['hashes'] == assinatura:
            return anterior['qualidade']
    resultados = []
    for item in itens:
        arquivo = ROOT/'data/bronze'/mes/item['arquivo']
        with tempfile.TemporaryDirectory() as tmp:
            with py7zr.SevenZipFile(arquivo, mode='r') as z:
                membros = z.getnames()
                if any(Path(n).is_absolute() or '..' in Path(n).parts for n in membros):
                    raise ValueError('Caminho inseguro dentro do arquivo compactado')
                z.extractall(path=tmp)
            textos = sorted(Path(tmp).rglob('*.txt'))
            if not textos:
                raise ValueError(f'Nenhum TXT em {arquivo.name}')
            for txt in textos:
                q = resumir_txt(txt, mes)
                q['arquivo_7z'] = item['arquivo']
                resultados.append(q)
    cache.write_text(json.dumps(dict(hashes=assinatura, qualidade=resultados), indent=2), encoding='utf-8')
    return resultados


In [6]:
linhas = []
for i, mes in enumerate(MESES, 1):
    print(f'[{i}/{len(MESES)}] Competência {mes}', flush=True)
    linhas.extend(processar_mes(mes))
qualidade = pd.DataFrame(linhas)
qualidade.to_csv(ROOT/'outputs/qualidade_por_arquivo.csv', index=False)
display(qualidade)
problemas = ['nulos_competencia','nulos_sinal','sinais_invalidos',
             'competencias_invalidas','competencia_divergente']
resumo_qualidade = qualidade[problemas].sum().to_frame('ocorrencias')
resumo_qualidade['percentual_registros'] = 100*resumo_qualidade.ocorrencias/qualidade.registros.sum()
display(resumo_qualidade)
assert qualidade.registros.sum() > 0, 'Nenhum registro encontrado.'
assert qualidade[problemas].sum().sum() == 0, 'Há inconsistências: revise qualidade_por_arquivo.csv antes de prosseguir.'
assert (qualidade.registros == qualidade.admissoes + qualidade.desligamentos).all()


[1/80] Competência 202001
[2/80] Competência 202002
[3/80] Competência 202003
[4/80] Competência 202004


KeyboardInterrupt: 

In [ ]:
serie = qualidade.groupby('mes')[['admissoes','desligamentos']].sum()
serie.index = pd.to_datetime(serie.index, format='%Y%m')
serie.index.name = 'competencia'
esperado = pd.date_range(INICIO, FIM, freq='MS')
assert serie.index.is_unique
assert serie.index.sort_values().equals(esperado), 'Existem competências ausentes ou adicionais.'
serie = serie.sort_index().asfreq('MS')
serie['saldo'] = serie.admissoes - serie.desligamentos
assert not serie.isna().any().any()
serie.to_csv(ROOT/'data/gold/serie_mensal_mov.csv')
serie.to_parquet(ROOT/'data/gold/serie_mensal_mov.parquet')
display(serie.head())
display(pd.Series({'inicio':str(serie.index.min().date()), 'fim':str(serie.index.max().date()),
                  'meses':len(serie), 'movimentacoes':int(qualidade.registros.sum()),
                  'meses_ausentes':0, 'recorte':'MOV sem FOR/EXC'}))


## 5. Divisão temporal antes de decisões de modelagem
Reservamos os últimos 12 meses para teste e não ajustamos o modelo usando esse período. A exploração usada para escolher a especificação será realizada no treinamento. O panorama completo é descritivo e não será usado para selecionar parâmetros. Não há embaralhamento, normalização global ou preenchimento com dados futuros.


In [ ]:
y = serie['saldo'].astype(float)
treino, teste = y.iloc[:-MESES_TESTE], y.iloc[-MESES_TESTE:]
assert len(treino) >= 36 and len(teste) == 12
display(pd.DataFrame({'inicio':[treino.index.min(),teste.index.min()],
                      'fim':[treino.index.max(),teste.index.max()],
                      'meses':[len(treino),len(teste)]}, index=['treino','teste']))
def salvar_fig(nome):
    plt.tight_layout()
    plt.savefig(ROOT/'outputs'/f'{nome}.png', dpi=160, bbox_inches='tight')
    plt.show()


## 6. Análise exploratória
### 6.1 Panorama descritivo
Os totais de admissão e desligamento mostram fluxos; o saldo é a diferença entre eles. Somar saldos não informa o estoque absoluto de vínculos sem um estoque inicial compatível. A região sombreada identifica o teste reservado.


In [ ]:
display(serie.describe().round(2))
fig, axs = plt.subplots(2, 1, figsize=(13,8), sharex=True)
serie[['admissoes','desligamentos']].plot(ax=axs[0], title='Movimentações nacionais — recorte MOV')
serie.saldo.plot(ax=axs[1], label='Saldo mensal', color='navy')
axs[1].axhline(0, color='black', lw=.8)
axs[1].set_title('Saldo mensal sem incorporação de FOR/EXC')
for ax in axs:
    ax.axvspan(teste.index[0], teste.index[-1], color='orange', alpha=.12, label='Teste reservado')
    ax.set_ylabel('Vínculos / movimentações')
    ax.legend()
salvar_fig('01_panorama')
display(serie.loc[[serie.saldo.idxmin(),serie.saldo.idxmax()]].rename_axis('competencia_extrema'))


### 6.2 Tendência, sazonalidade e dependência no treinamento
A média móvel é retrospectiva. O mapa de calor permite comparar meses entre anos; anos incompletos permanecem com células vazias. A ACF descreve associação temporal, sem estabelecer causalidade. A decomposição STL é exploratória e não é utilizada como entrada do modelo.


In [ ]:
fig, ax = plt.subplots()
treino.plot(ax=ax, alpha=.65, label='Saldo no treino')
treino.rolling(12, min_periods=12).mean().plot(ax=ax, lw=2, label='Média móvel de 12 meses')
ax.legend(); ax.set_title('Evolução no treinamento'); ax.set_ylabel('Saldo de vínculos')
salvar_fig('02_tendencia_treino')
explora = treino.rename('saldo').to_frame().assign(ano=treino.index.year, mes=treino.index.month)
fig, ax = plt.subplots(figsize=(12,5))
sns.heatmap(explora.pivot(index='ano',columns='mes',values='saldo'), center=0, cmap='RdBu', ax=ax)
ax.set_title('Saldo por ano e mês — somente treino')
salvar_fig('03_sazonalidade')
fig, ax = plt.subplots()
plot_acf(treino, lags=min(24,len(treino)//2-1), ax=ax)
salvar_fig('04_acf')
decomp = STL(treino, period=12, robust=True).fit()
decomp.plot(); salvar_fig('05_stl')
residuo_stl = decomp.resid
mediana = residuo_stl.median()
mad = (residuo_stl-mediana).abs().median()
if mad > 0:
    score = .6745*(residuo_stl-mediana)/mad
    candidatos = pd.DataFrame({'saldo':treino,'residuo_stl':residuo_stl,'score_robusto':score})
    display(candidatos.loc[score.abs()>3.5])
else:
    print('MAD zero: sinalização robusta não calculada.')


**Interpretação a concluir após a execução:** quais meses apresentam padrões recorrentes? Há alterações de nível? Quais competências têm maior saldo absoluto? Os candidatos identificados por resíduos STL são apenas sinais exploratórios, não anomalias confirmadas. Não removemos observações: choques reais fazem parte do fenômeno. Explicações sobre eventos externos exigem fontes adicionais.

**Limitações:** série curta, início em 2020, poucas repetições anuais, mudanças econômicas, ausência de variáveis externas, revisões históricas e exclusão de FOR/EXC. O recorte nacional pode esconder diferenças setoriais e regionais. Não há evidência suficiente aqui para inferência causal.


## 7. Primeiro experimento de previsão
**Desenho:** origem única no fim do treinamento, horizonte de 12 meses, sem atualizar o modelo com valores observados no teste.

**Baseline sazonal:** para cada mês, repetir o valor do mesmo mês do ano anterior.  
**Modelo inicial:** Holt-Winters com tendência aditiva amortecida e sazonalidade aditiva de período 12. A formulação aditiva admite saldos negativos. A especificação é fixada antes da avaliação; a estimação dos parâmetros usa somente o treino. Este é um teste inicial, sem busca de hiperparâmetros. Se a sazonalidade não for sustentada pela exploração, registre isso como limitação e planeje uma especificação alternativa com validação no treino.

MAE mede o erro absoluto médio na unidade do saldo; RMSE dá mais peso a erros grandes. MAPE não é utilizada porque o saldo pode ser zero ou negativo.


In [ ]:
def ajustar_hw(dados):
    return ExponentialSmoothing(dados, trend='add', damped_trend=True,
                                seasonal='add', seasonal_periods=12,
                                initialization_method='estimated').fit(optimized=True)

pred_baseline = pd.Series(treino.iloc[-12:].to_numpy(), index=teste.index, name='Baseline sazonal')
with warnings.catch_warnings(record=True) as avisos:
    warnings.simplefilter('always')
    modelo = ajustar_hw(treino)
    pred_hw = modelo.forecast(len(teste)).rename('Holt-Winters')
    for aviso in avisos:
        print(f'AVISO DO AJUSTE: {aviso.message}')
assert pred_hw.index.equals(teste.index)
assert np.isfinite(pred_hw).all()
display(pd.Series(modelo.params, name='parametros_estimados'))
print('Informações de otimização:', modelo.mle_retvals)
previsoes = pd.concat([teste.rename('real'),pred_baseline,pred_hw],axis=1)

def metricas(real, previsto):
    erro = real-previsto
    return {'MAE':float(np.abs(erro).mean()), 'RMSE':float(np.sqrt(np.mean(erro**2)))}

resultados = pd.DataFrame({nome:metricas(teste,previsoes[nome])
                          for nome in ['Baseline sazonal','Holt-Winters']}).T
display(resultados.round(2))
resultados.to_csv(ROOT/'outputs/metricas_teste.csv')
previsoes.to_csv(ROOT/'outputs/previsoes_teste.csv')
previsoes.plot(marker='o', title='Teste temporal de 12 meses — saldo do recorte MOV')
plt.ylabel('Saldo de vínculos'); salvar_fig('06_previsoes_teste')


In [ ]:
erros = previsoes[['Baseline sazonal','Holt-Winters']].rsub(previsoes.real,axis=0)
display(erros.abs().rename(columns=lambda c:'erro_absoluto_'+c).nlargest(5,'erro_absoluto_Holt-Winters'))
erros.plot(marker='o', title='Erros de previsão no teste: real − previsto')
plt.axhline(0,color='black',lw=.8); salvar_fig('07_erros_teste')
residuos = pd.Series(modelo.resid, index=treino.index).iloc[12:]
display(acorr_ljungbox(residuos, lags=[min(12,len(residuos)//3)], model_df=0, return_df=True))
print('Ljung–Box acima é diagnóstico exploratório no treino; não comprova adequação do modelo.')
base_mae = resultados.loc['Baseline sazonal','MAE']
hw_mae = resultados.loc['Holt-Winters','MAE']
if base_mae > 0:
    ganho = 100*(base_mae-hw_mae)/base_mae
    display(Markdown(f'**Resultado calculado:** variação relativa do MAE do Holt-Winters frente ao baseline: **{ganho:.2f}%** de redução (valor negativo indica piora).'))
else:
    print('MAE do baseline igual a zero; redução percentual não definida.')
display(Markdown('O resultado é restrito a este intervalo de teste. Não demonstra superioridade geral nem constitui validação em tempo real.'))


## 8. Discussão dos resultados parciais
Após executar, redija uma síntese própria utilizando os números acima:
1. Informe o intervalo, número de meses e quantidade de registros efetivamente utilizados.
2. Descreva dois padrões observados nos gráficos e as competências que os sustentam.
3. Compare MAE e RMSE das duas abordagens; se apontarem conclusões diferentes, explique o peso de erros grandes no RMSE.
4. Identifique os meses de maior erro e discuta possíveis limitações sem atribuir causas não verificadas.
5. Registre se houve avisos ou problemas de convergência. Não trate um ajuste que falhou como resultado válido.
6. Explique por que este experimento ainda não responde pela previsão do saldo oficial ajustado.

Um modelo que perde para o baseline é um resultado parcial legítimo. A entrega deve explicar o observado e propor um próximo teste, sem inventar ganhos. Previsões futuras após reajuste em toda a série ficam para a próxima etapa; este notebook prioriza a avaliação fora da amostra.


## 9. Pipeline atualizado e próximos passos
| Etapa | Implementação nesta versão | Próxima evolução |
|---|---|---|
| Bronze | Arquivos MOV originais, hashes e manifesto | Incorporar snapshots e revisão da fonte |
| Validação | Esquema, domínio, nulos, competência e continuidade | Conciliação com totais oficiais de mesma versão |
| Silver | Contagens por arquivo/mês e diagnóstico | Tratar FOR/EXC e recortes por UF/setor |
| Gold | Série nacional mensal em CSV/Parquet | Saldo ajustado e metadados de revisão |
| Exploração | Evolução, média móvel, sazonalidade, ACF e STL | Investigar eventos e recortes específicos |
| Modelagem | Baseline e Holt-Winters inicial | Validação com origens móveis; SARIMA se justificado |
| Avaliação | MAE/RMSE e erros em teste de 12 meses | Robustez por horizonte e intervalos de previsão |

Fluxo: **MTE → arquivos MOV → Bronze → validação → Silver agregada → Gold mensal → exploração no treino → baseline/Holt-Winters → teste temporal → discussão.**

As rotinas acima estão implementadas no código; sua execução com dados reais deve ser confirmada pelas saídas. Não há agendamento automático de ingestão nesta etapa. XGBoost permanece opcional.

## 10. Cronograma revisado
Checkpoint informado: **05/10/2026**. Entrega parcial: **26/10/2026**. Datas abaixo são planejamento; atualizar status conforme a execução.

| Período | Atividade | Evidência esperada | Status inicial |
|---|---|---|---|
| 18–19/10 | Coleta e verificação da cobertura | Manifestos e arquivos | A executar no Colab |
| 20/10 | Tratamento e agregação | Qualidade e série mensal | A executar |
| 21/10 | Exploração e interpretação | Gráficos e texto analítico | A executar |
| 22–23/10 | Baseline e modelo estatístico | Previsões e diagnósticos | A executar |
| 24/10 | Comparação e discussão | MAE/RMSE e análise dos erros | A executar |
| 25/10 | Revisão e execução integral | Notebook salvo com saídas | A executar |
| 26/10 | Entrega da Etapa 3 | Notebook e repositório | Planejado |
| Após 26/10 | Ajustes FOR/EXC, validação móvel e refinamento | Experimentos documentados | Entrega final a confirmar |

## 11. Checklist para entrega
- [ ] Todas as células executadas em ordem e salvas com saídas.
- [ ] Origem, período e recorte MOV explicitados.
- [ ] Qualidade verificada e problemas tratados sem exclusões silenciosas.
- [ ] Gráficos interpretados pelo grupo.
- [ ] Baseline e modelo inicial avaliados nos mesmos meses.
- [ ] Discussão preenchida com os resultados efetivos.
- [ ] Cronograma e pipeline refletem o estado real.
- [ ] Repositório atualizado, sem microdados volumosos ou credenciais.

## 12. Referências e materiais de apoio
- [MTE — Microdados RAIS e CAGED](https://www.gov.br/trabalho-e-emprego/pt-br/assuntos/estatisticas-trabalho/microdados-rais-e-caged).
- [MTE — Nota técnica sobre exclusões](https://www.gov.br/trabalho-e-emprego/pt-br/assuntos/estatisticas-trabalho/novo-caged/novo-caged-2023/outubro/nota_tecnica_novo_caged_11-2021.pdf).
- [Base dos Dados — implementação de referência para coleta e nomes dos campos](https://github.com/basedosdados/queries-basedosdados-dev/blob/main/models/br_me_caged/code/crawler_caged.py). Consultada para estrutura; os dados deste notebook vêm do MTE.
- [Hyndman e Athanasopoulos — Forecasting: Principles and Practice](https://otexts.com/fpp3/).
- [Statsmodels — ExponentialSmoothing](https://www.statsmodels.org/stable/generated/statsmodels.tsa.holtwinters.ExponentialSmoothing.html).
- Documento do grupo: Projeto Aplicado IV A2 V1.
- [Repositório do projeto](https://github.com/freitasbtt/p4_mackenzie_grupo_11).
